# ⚖️ Rechtsprechungs-Agent — POC

Durchsucht automatisch **4 deutsche Gerichtsdatenbanken** und erstellt eine juristische Zusammenfassung mit **Claude (Anthropic)**.

| Schritt | Zelle | Dauer |
|---------|-------|-------|
| 1 | Installation | ~3 Min. |
| 2 | ZIP hochladen | ~1 Min. |
| 3 | API-Key eintragen | sofort |
| 4 | Suche starten | ~1–2 Min. |

▶️ **Jede Zelle einzeln ausführen — von oben nach unten.**

---
## Schritt 1: Installation

Installiert Playwright (Browser-Automatisierung) und das Anthropic SDK.
**Dauert ca. 2–3 Minuten** — bitte warten bis ✅ erscheint.

In [ ]:
# Pakete installieren
!pip install playwright anthropic --quiet

# Chromium-Browser herunterladen (wird für das Scraping benötigt)
!playwright install chromium
!playwright install-deps chromium

print("\n✅ Installation abgeschlossen.")

---
## Schritt 2: ZIP-Datei hochladen und entpacken

Führen Sie die Zelle aus. Es erscheint ein **Upload-Button**.
Laden Sie die Datei `rechtsprechung_agent_poc.zip` hoch.

In [ ]:
from google.colab import files
import zipfile, os

print("Bitte ZIP-Datei auswählen ...")
hochgeladen = files.upload()

# ZIP entpacken
for dateiname in hochgeladen.keys():
    with zipfile.ZipFile(dateiname, 'r') as z:
        z.extractall('.')
    print(f"✅ '{dateiname}' entpackt.")

# In das Projektverzeichnis wechseln
os.chdir('rechtsprechung_agent')
print(f"📁 Arbeitsverzeichnis: {os.getcwd()}")
print("\nDateien im Projekt:")
for f in sorted(os.listdir('.')):
    print(f"  {f}")

---
## Schritt 3: Anthropic API-Key eintragen

Tragen Sie Ihren API-Key ein. Den Key finden Sie unter:
👉 https://console.anthropic.com/

⚠️ **Wichtig**: Teilen Sie dieses Notebook niemals mit eingetragenem Key.

In [ ]:
import os
from getpass import getpass

# Sicheres Eingabefeld (Key wird nicht angezeigt)
api_key = getpass("Anthropic API-Key eingeben (sk-ant-...): ")
os.environ["ANTHROPIC_API_KEY"] = api_key

# Kurzer Funktionstest
import anthropic
try:
    client = anthropic.Anthropic(api_key=api_key)
    test = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=10,
        messages=[{"role": "user", "content": "Antworte nur: OK"}]
    )
    print("✅ API-Key gültig. Verbindung zu Claude erfolgreich.")
except Exception as e:
    print(f"❌ Fehler: {e}")
    print("Bitte API-Key prüfen.")

---
## Schritt 4: Suche starten

Tragen Sie Ihren Suchbegriff ein und führen Sie die Zelle aus.

**Beispiele:**
- `Mietminderung Schimmel`
- `Kündigung fristlos`
- `Werkvertrag Mängel`
- `Datenschutz DSGVO`

In [ ]:
# ── Suchbegriff anpassen ──────────────────────────────────────────
SUCHBEGRIFF = "Mietminderung Schimmel"

# Optional: Zeitraum einschränken (Format: TT.MM.JJJJ oder None)
DATUM_VON = None   # z.B. "01.01.2022"
DATUM_BIS = None   # z.B. "31.12.2024"

# Optional: Nur bestimmte Portale (None = alle 4)
# Möglich: ["Brandenburg", "NRW", "Bayern", "Niedersachsen"]
NUR_PORTALE = None
# ─────────────────────────────────────────────────────────────────

import asyncio
import sys
sys.path.insert(0, '.')

from main import agent

# Suche ausführen
ergebnis = await agent(
    suchbegriff=SUCHBEGRIFF,
    nur_portale=NUR_PORTALE,
    datum_von=DATUM_VON,
    datum_bis=DATUM_BIS,
)

print(ergebnis)

---
## Schritt 5: Ergebnis herunterladen (optional)

Lädt die gespeicherte Zusammenfassung als `.txt`-Datei herunter.

In [ ]:
import os, glob
from google.colab import files

# Neueste Ausgabedatei finden
ausgaben = sorted(glob.glob('output/zusammenfassung_*.txt'))

if ausgaben:
    neueste = ausgaben[-1]
    print(f"Lade herunter: {neueste}")
    files.download(neueste)
else:
    print("Keine Ausgabedatei gefunden. Bitte zuerst Schritt 4 ausführen.")

---
## Fehlerbehebung

**Leere Trefferliste?**
Die CSS-Selektoren der Portale könnten sich geändert haben.
Führen Sie diese Zelle aus, um die rohe HTML-Struktur zu prüfen:

In [ ]:
# Diagnose-Zelle: Zeigt HTML-Struktur der Suchergebnisse
# Nur bei Problemen ausführen

import asyncio
from playwright.async_api import async_playwright
from urllib.parse import quote

TEST_SUCHBEGRIFF = SUCHBEGRIFF
TEST_PORTAL = "Bayern"  # oder: Brandenburg, NRW, Niedersachsen

URLS = {
    "Bayern":        f"https://www.gesetze-bayern.de/Search?query={quote(TEST_SUCHBEGRIFF)}&publicationtype=publicationform-ats-filter!ATS_Rechtsprechung",
    "Niedersachsen": f"https://voris.wolterskluwer-online.de/search?query={quote(TEST_SUCHBEGRIFF)}&publicationtype=publicationform-ats-filter!ATS_Rechtsprechung",
    "Brandenburg":   "https://gerichtsentscheidungen.brandenburg.de/suche",
    "NRW":           "https://nrwesuche.justiz.nrw.de/",
}

async def diagnose():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto(URLS[TEST_PORTAL], wait_until="networkidle")
        html = await page.content()
        await browser.close()
    # Ersten 3000 Zeichen des Body ausgeben
    start = html.find('<body')
    print(f"--- HTML-Ausschnitt ({TEST_PORTAL}) ---")
    print(html[start:start+3000])

await diagnose()